### Donwload MIMIC MEDS Dataset

In [ ]:
!conda create -y -n venv_meds_mimic python=3.11

In [ ]:
cohort = "MEDS_cohort"
do_download = "True"
# physionet credentials
username = ""
password = ""


!conda run -n venv_meds_mimic bash run_meds_etl.sh \
"{cohort}" "{username}" "{password}" "{do_download}"

## 2) Generate prediction labels for each task

In [ ]:
import os

TASKS_PATH = "tasks"
MEDS_cohort = "MEDS_cohort"

for f in os.listdir(TASKS_PATH):
    f_name = os.path.splitext(f)[0]
    !aces-cli \
        config_path="{TASKS_PATH}/{f}" \
        cohort_name="{f_name}" \
        cohort_dir="{MEDS_cohort}/labels" \
        data=sharded \
        data.standard=meds \
        data.root="{MEDS_cohort}/data" \
        data.shard=$(expand_shards train/292 tuning/37 held_out/37) \
        -m

## 3) Preprocessing pipeline & MEDS-KG conversion

In [ ]:
import yaml
import os
import polars as pl


def init_dirs(task_name: str, index: int, root = "exports"):
    export_dir = f"{root}/{task_name}"
    outcomes_dir = f"{export_dir}/labels"
    meds_cohort_dir = f"{export_dir}/meds/{index}/MEDS_cohort"
    os.makedirs(export_dir, exist_ok=True)
    os.makedirs(outcomes_dir, exist_ok=True)
    os.makedirs(meds_cohort_dir, exist_ok=True)
    os.makedirs(f"{export_dir}/meds/{index}/MEDS_cohort/metadata", exist_ok=True)

    return export_dir, outcomes_dir, meds_cohort_dir


with open("experiments.yaml", "r") as f:
    config = yaml.safe_load(f)


def iter_tasks(config):
    for group_name, experiments in config["experiments"].items():
        for scfg in experiments:
            yield group_name, scfg

EXP = "exp-1k"

### 3.1) Process MIMIC-MEDS dataset and create 11 task samples

In [ ]:
from utils.preprocessing import (
    remove_long_text,
    birthdate_to_age,
    window_dict,
    parse_blood_pressure_values,
    regenerate_ids,
    meds_core_columns,
)


def build_events_base(events, sample, scfg):
    base = (
        events.join(sample.lazy(), on="subject_id", how="inner")
        .with_columns(remove_long_text)
        .with_columns(birthdate_to_age)
        .filter(window_dict[scfg["window"]])
    )

    base = (
        parse_blood_pressure_values(base)
        .with_columns(regenerate_ids)
        .select(meds_core_columns)
    )

    return base


def split_event_types(events_base, scfg):
    static_codes = (
        events_base.filter(pl.col("time").is_null() & pl.col("numeric_value").is_null())
        .sort(["subject_id", "code"])
        .unique(subset=["subject_id", "code"], keep="last")
    )

    static_numeric = (
        events_base.filter(
            pl.col("time").is_null() & pl.col("numeric_value").is_not_null()
        )
        .sort(["subject_id", "code"])
        .unique(subset=["subject_id", "code"], keep="last")
    )

    dynamic_numeric = (
        events_base.filter(
            pl.col("time").is_not_null() & pl.col("numeric_value").is_not_null()
        )
        .with_columns(pl.col("time").dt.truncate(scfg["agg"]).alias("time"))
        .sort(["subject_id", "code", "time"])
        .group_by(["subject_id", "code", "time"])
        .agg(
            [
                pl.col("numeric_value").min().alias("min"),
                pl.col("numeric_value").max().alias("max"),
                pl.col("prediction_time").last(),
                pl.col("boolean_value").last(),
            ]
        )
    )

    dynamic_numeric = (
        dynamic_numeric.unpivot(
            index=["subject_id", "code", "time", "prediction_time", "boolean_value"],
            on=["min", "max"],
            variable_name="stat",
            value_name="numeric_value",
        )
        .with_columns(
            pl.concat_str([pl.col("code"), pl.lit("_"), pl.col("stat")]).alias("code")
        )
        .drop("stat")
        .select(meds_core_columns)
    ).filter(pl.col("numeric_value").is_not_null())

    # dynamic_numeric = aggregate_events(dynamic_numeric, agg=scfg["agg"])

    dynamic_codes = events_base.filter(
        pl.col("time").is_not_null() & pl.col("numeric_value").is_null()
    )

    return static_codes, static_numeric, dynamic_numeric, dynamic_codes

In [ ]:
from pathlib import Path

from utils.preprocessing import generate_samples, filter_by_treshold, create_meds_cohort
import joblib
from utils.preprocessing import PREFIX_MAP, process_codes

events = pl.scan_parquet("MEDS_cohort/data/**/*.parquet", low_memory=True).select(
    "subject_id", "time", "code", "numeric_value", "text_value"
)

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")

    IHM = f"MEDS_cohort/labels/{scfg['task']}/**/*.parquet"

    outcomes = pl.scan_parquet(IHM).select(
        "subject_id", "prediction_time", "boolean_value"
    )

    samples = generate_samples(
        labels=outcomes,
        n_folds=scfg["num_of_samples"],
        size=scfg["sample_size"],
        seed=1234,
        low_true_values=scfg["fixed_true"],
    )

    TRESHOLD = scfg["sample_size"] / 5
    print("TRESHOLD: > ", TRESHOLD)

    for index, sample in enumerate(samples):
        export_dir, outcomes_dir, meds_cohort_dir = init_dirs(scfg["task"], index, root=EXP)

        events_base = build_events_base(events, sample, scfg)

        static_codes, static_numeric, dynamic_numeric, dynamic_codes = (
            split_event_types(events_base, scfg)
        )

        final_events = filter_by_treshold(
            pl.concat(
                [
                    static_numeric,
                    static_codes,
                    dynamic_numeric,
                    dynamic_codes,
                ],
            ),
            TRESHOLD,
        ).collect(engine="streaming")

        print(f"TOTAL EVENTS: {len(final_events)}")
        print(f"TOTAL CODES: {len(final_events.group_by('code').len())}")

        (_, _, split_l) = create_meds_cohort(
            final_events,
            orig_dir="MEDS_cohort",
            output_dir=meds_cohort_dir,
            columns=["subject_id", "code", "time", "numeric_value"],
        )

        joblib.dump(
            value=split_l.sort("subject_id")["boolean_value"].to_numpy(),
            filename=f"{outcomes_dir}/outcomes_meds_TS_{scfg['sample_size']}_{index}.joblib",
        )

        process_codes(
            input_parquet=f"{meds_cohort_dir}/metadata/codes.parquet",
            output_dir=f"{meds_cohort_dir}/mimic_external_codes",
            prefix_map=PREFIX_MAP,
        )

### 3.2) Predict patient outcomes with tabular-based models

In [ ]:
from utils.tabular import run_tabulars_models
import yaml
import joblib
import shutil

CLASSES = ["FALSE", "TRUE"]

with open("experiments.yaml", "r") as f:
    config = yaml.safe_load(f)

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")

    ETL_LABELS = f"{EXP}/{scfg['task']}/labels"

    for i in range(scfg["num_of_samples"]):
        EXPORT_DIR = f"{EXP}/{scfg['task']}/meds/{i}"
        try:
            shutil.rmtree(f"{EXPORT_DIR}/metrics_1000")
        except FileNotFoundError:
            print("File not found")
        
        X, y = run_tabulars_models(
            meds_root=f"{EXPORT_DIR}/MEDS_cohort",
            classes=CLASSES,
            outcomes_path=f"{ETL_LABELS}/outcomes_meds_TS_{scfg['sample_size']}_{i}.joblib",
            result_dir=f"{EXPORT_DIR}/metrics_{scfg['sample_size']}",
            save_model=True,
        )

        joblib.dump(
            X.columns.to_list(),
            f"{EXPORT_DIR}/metrics_{scfg['sample_size']}/feature_names.joblib",
        )
        joblib.dump(X, f"{EXPORT_DIR}/metrics_{scfg['sample_size']}/X.joblib")

### 3.3) Explain results through SHAP

In [ ]:
from rdflib import Namespace
import json
from pathlib import Path
import joblib

import shap

from utils.shap_explainer import sanitize_for_uri, save_best_features, save_top_shap_features_per_class

NS_DATA = Namespace("https://teamheka.github.io/meds-data/")
NS_ONTO = Namespace("https://teamheka.github.io/meds-ontology#")
NS_CODE = Namespace(f"{NS_DATA}code/")

MODEL = "xgboost"
QUANTILE  = 0.9
CLASSES = ["TRUE", "FALSE"]

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")

    for i in range(scfg["num_of_samples"]):
        EXPORT_DIR = f"{EXP}/{scfg['task']}/meds/{i}/metrics_{scfg['sample_size']}"

        feature_names = joblib.load(f"{EXPORT_DIR}/feature_names.joblib")
        X = joblib.load(f"{EXPORT_DIR}/X.joblib")

        model_dir = Path(f"{EXPORT_DIR}/{MODEL}/models")

        print(str(model_dir))

        best_model_path = str(
            max(
                model_dir.glob(f"{MODEL}_best_fold*_auc_*.joblib"),
                key=lambda p: float(p.stem.split("_auc_")[-1]),
            )
        )

        explainer = shap.TreeExplainer(model=joblib.load(best_model_path))

        save_best_features(
            output_dir=f"{EXPORT_DIR}/{MODEL}",
            explainer=explainer,
            feature_names=feature_names,
            X=X,
            classes=CLASSES
        )

        top_features_per_class = save_top_shap_features_per_class(
            explainer=explainer,
            X=X,
            feature_names=feature_names,
            classes=CLASSES,
            output_dir=f"{EXPORT_DIR}/{MODEL}",
            quantile=QUANTILE
        )


        all_features = []
        for class_name, content in top_features_per_class.items():
            all_features.extend(content["features"])

        mimic_dict = {
            NS_CODE[sanitize_for_uri(feature_name)]: NS_ONTO[
                f"has{sanitize_for_uri(feature_name)}"
            ]
            for feature_name in set(all_features)
        }

        with open(f"{EXPORT_DIR}/{MODEL}/onto_features_dict_{QUANTILE}.json", "w") as f:
            json.dump(mimic_dict, f, indent=2)

### 3.4) Filter by best features

In [ ]:
import numpy as np

from utils.shap_explainer import clean_xgb_feature_name
from utils.preprocessing import generate_samples, filter_by_treshold, create_meds_cohort
import joblib
from utils.preprocessing import PREFIX_MAP, process_codes

OUTPUT = f"{EXP}-{QUANTILE}"

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")


    for index in range(0, scfg["num_of_samples"]):
        EXPORT = f"{EXP}/{scfg['task']}/meds/{index}"

        outcomes = pl.read_parquet(f"{EXPORT}/MEDS_cohort/labels/**/*.parquet").select(
            "subject_id", "prediction_time", "boolean_value"
        )

        arr = np.load(f"{EXPORT}/metrics_{scfg["sample_size"]}/xgboost/features_{QUANTILE}.npy", allow_pickle=True)
        export_dir, outcomes_dir, meds_cohort_dir = init_dirs(scfg["task"], index, root=OUTPUT)

        events = pl.scan_parquet(f"{EXPORT}/MEDS_cohort/data/**/*.parquet", low_memory=True).collect(engine="streaming")
        print(len(events))

        events = events.filter(
            pl.col("code").map_elements(clean_xgb_feature_name, return_dtype=pl.Utf8).is_in(arr)
        )
        print(len(events))

        events = events.join(outcomes, on="subject_id", how="inner")

        (_, _, split_l) = create_meds_cohort(
            events,
            orig_dir="MEDS_cohort",
            output_dir=meds_cohort_dir,
            columns=["subject_id", "code", "time", "numeric_value"],
        )

        joblib.dump(
            value=split_l.sort("subject_id")["boolean_value"].to_numpy(),
            filename=f"{outcomes_dir}/outcomes_meds_TS_{scfg['sample_size']}_{index}.joblib",
        )

        process_codes(
            input_parquet=f"{meds_cohort_dir}/metadata/codes.parquet",
            output_dir=f"{meds_cohort_dir}/mimic_external_codes",
            prefix_map=PREFIX_MAP,
        )


### 3.5) Convert task samples into MEDS-KG

In [ ]:
from meds2rdf import MedsRDFConverter, NTriplesSink, Config, MEDSSchema
import shutil
from pathlib import Path

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")

    for index in range(scfg["num_of_samples"]):
        _, _, meds_cohort_dir = init_dirs(scfg["task"], index, root="exp0-20k-0.97")
        export_dir, _, _ = init_dirs(scfg["task"], index, root="exp3-20k-0.97")

        #shutil.rmtree(f"{export_dir}/meds_{scfg['sample_size']}_{index}")

        MedsRDFConverter(meds_cohort_dir).convert(
            sink=NTriplesSink(
                Path(f"{export_dir}/meds_{scfg['sample_size']}_{index}"),
                gzip_mode=False,
            ),
            cfg=Config(schemas={MEDSSchema.CODES}),
        )

### 3.6) Predict patient outcomes with tabular-based models on filtered dataset

In [ ]:
from utils.tabular import run_tabulars_models
import yaml
import joblib
import shutil

CLASSES = ["FALSE", "TRUE"]

with open("experiments.yaml", "r") as f:
    config = yaml.safe_load(f)

for group_name, scfg in iter_tasks(config):
    print(f"\n=== Group: {group_name} ===")
    print(f"Running task: {scfg['task']}")

    ETL_LABELS = f"{OUTPUT}/{scfg['task']}/labels"

    for i in range(scfg["num_of_samples"]):
        EXPORT_DIR = f"{OUTPUT}/{scfg['task']}/meds/{i}"
        try:
            shutil.rmtree(f"{EXPORT_DIR}/metrics_1000")
        except FileNotFoundError:
            print("File not found")
        
        X, y = run_tabulars_models(
            meds_root=f"{EXPORT_DIR}/MEDS_cohort",
            classes=CLASSES,
            outcomes_path=f"{ETL_LABELS}/outcomes_meds_TS_{scfg['sample_size']}_{i}.joblib",
            result_dir=f"{EXPORT_DIR}/metrics_{scfg['sample_size']}",
            save_model=True,
        )

        joblib.dump(
            X.columns.to_list(),
            f"{EXPORT_DIR}/metrics_{scfg['sample_size']}/feature_names.joblib",
        )
        joblib.dump(X, f"{EXPORT_DIR}/metrics_{scfg['sample_size']}/X.joblib")

### 3.6) Compute Tabular AVGs on filtered dataset

In [ ]:
import yaml
from pathlib import Path
from pathlib import Path

from utils.tabular import aggregate_task, format_row

MODEL = "xgboost"
BASE_RESULTS = Path(OUTPUT)

# -------------------------
# LOAD EXPERIMENT CONFIG
# -------------------------
with open("experiments.yaml", "r") as f:
    exp_config = yaml.safe_load(f)["experiments"]


all_results = {}

# -------------------------
# LOOP OVER TASKS
# -------------------------
for group_name, experiments in exp_config.items():
    print(f"\n=== Group: {group_name} ===")

    for exp in experiments:
        task_name = exp["task"]
        print(f"Running task: {task_name}")

        summary = aggregate_task(exp, results_dir=BASE_RESULTS, model_type=MODEL)

        all_results[task_name] = summary

        # pretty print
        print("Result:")
        format_row(summary)